In [1]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:

from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer, default_data_collator
import evaluate
import numpy as np
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor
from PIL import Image
import torch


food = load_dataset("food101", split="train[:5000]")
food_split = food.train_test_split(test_size=0.2)


labels = food_split['train'].features['label'].names
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}


processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
size = processor.size["shortest_edge"]


transform = Compose([
    RandomResizedCrop(size),
    ToTensor(),
    Normalize(mean=processor.image_mean, std=processor.image_std)
])


def process_image(samples):
    samples['pixel_values'] = [transform(img.convert('RGB')) for img in samples['image']]
    del samples['image']
    return samples

food_split = food_split.with_transform(process_image)


model = AutoModelForImageClassification.from_pretrained(
    "microsoft/resnet-50",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(

    output_dir="./results",
    report_to=[],
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    weight_decay=0.01,
    remove_unused_columns=False,
    save_strategy="epoch",
    learning_rate=0.001,
    num_train_epochs=7,
    logging_steps=10,
    push_to_hub=False,
    metric_for_best_model="accuracy"
)


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    train_dataset=food_split['train'],
    eval_dataset=food_split['test'],
    processing_class=processor
)


trainer.train()

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Some weights of ResNetForImageClassification were not initialized from the model checkpoint at microsoft/resnet-50 and are newly initialized because the shapes did not match:
- classifier.1.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([101]) in the model instantiated
- classifier.1.weight: found shape torch.Size([1000, 2048]) in the checkpoint and torch.Size([101, 2048]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.803000,0.684820,0.772000
2,0.687300,0.555202,0.822000
3,0.547300,0.489121,0.836000
4,0.399000,0.457608,0.850000
5,0.453200,0.460263,0.846000
6,0.350200,0.393189,0.864000
7,0.294000,0.383634,0.876000


TrainOutput(global_step=1750, training_loss=0.5330646373885018, metrics={'train_runtime': 525.3805, 'train_samples_per_second': 53.295, 'train_steps_per_second': 3.331, 'total_flos': 5.99721134874624e+17, 'train_loss': 0.5330646373885018, 'epoch': 7.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/model_CV/resnet50")